In [1]:
# DS501 Project #4: Predictive Modeling Competition

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, f1_score, accuracy_score, classification_report
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.linear_model import LinearRegression, SGDRegressor, LogisticRegression
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer # Added missing import
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings in the notebook output
import warnings
warnings.filterwarnings('ignore')

# --- TASK 1: DATA PREPROCESSING AND EDA 

data_regression = pd.read_csv('AmesHousing.csv')
data_classification = pd.read_csv('defaultofcreditcardclients.csv')

# --- Helper Function for Data Preparation ---
def prepare_data(data, target_column, is_regression=True):
    # Separate target variable
    X = data.drop(columns=[target_column])
    y = data[target_column]
    
    # Identify feature types (Placeholder - adjust columns based on actual data)
    numerical_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(include='object').columns.tolist()
    
    # Create preprocessing pipelines
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        # MANDATORY for KNN and SVM (Task 1, Part 2)
        ('scaler', StandardScaler()) 
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    
    # Create column transformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='passthrough' # Keep any unlisted columns
    )
        
    return X, y, preprocessor


# UNCOMMENT AND EXECUTE THESE BLOCKS ONCE YOUR DATA IS LOADED.

# --- TASK 1.1: REGRESSION SETUP (Ames Housing) ---
print("--- Starting Regression Task Setup (Ames Housing) ---")
X_reg, y_reg, preprocessor_reg = prepare_data(data_regression, 'SalePrice')
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
X_reg, y_reg, test_size=0.5, random_state=42) # 50/50 Split (Task 3, Part 1)

# --- TASK 1.2: CLASSIFICATION SETUP (Credit Card Default) ---
print("--- Starting Classification Task Setup (Credit Card Default) ---")
X_clf, y_clf, preprocessor_clf = prepare_data(data_classification, 'default_payment_next_month', is_regression=False)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.5, random_state=42) # 50/50 Split (Task 3, Part 1)

# TASK 2 — MODELS

# --------------------------
# REGRESSION MODELS
# --------------------------

#first create a dictionary of keys being the model names and the values being the actual models
#I did this so it can loop through them automatically.
models_regression = {
    "Linear Regression :Normal Equation": LinearRegression(),
    "Linear Regression :Gradient Descent": SGDRegressor(max_iter=1000, tol=1e-3),
    "Decision Tree Regressor": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=7)
}

#create dic fro saving results: F1 score and accuracy
results_regression = {}
#then loop through the model

for name, model in models_regression.items():
    #pipe to preprocess the data, standarizes numerical values, one-hot encodes categorical values
    #it also imputes missing values, an then applies the model.
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor_reg),
        ("model", model)
    ])
    pipe.fit(X_train_reg, y_train_reg) #this trains the model on the training data
    #here it predicts
    preds = pipe.predict(X_test_reg)

    #calculates metrics
    rmse = np.sqrt(mean_squared_error(y_test_reg, preds))
    results_regression[name] = rmse
    print(f"{name}: RMSE = {rmse:.2f}")


# --------------------------
# CLASSIFICATION MODELS
# --------------------------

#first create a dictionary of keys being the model names and the values being the actual models
#I did this so it can loop through them automatically.

models_classification = {
    "Logistic regression": LogisticRegression(max_iter=500),
    "Decision tree Classifier": DecisionTreeClassifier(),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=200),
    "KNN Classifier": KNeighborsClassifier(n_neighbors=7),
    "SVM Classifier ": SVC()
}

results_classification = {}

for name, model in models_classification.items():
    #pipe to preprocess the data, standarizes numerical values, one-hot encodes categorical values
    #it also imputes missing values, an then applies the model.
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor_clf),
        ("model", model)
    ])

    pipe.fit(X_train_clf, y_train_clf)
    preds = pipe.predict(X_test_clf)

    accuracy = accuracy_score(y_test_clf, preds)
    f1 = f1_score(y_test_clf, preds)

    results_classification[name] = (accuracy, f1)

    print(f"{name}: Accuracy= {accuracy:.4f}, F1 = {f1:.4f}")


# TASK 3 — RESULTS TABLES

print("\n================= REGRESSION RESULTS (RMSE) =================")
reg_results = pd.DataFrame.from_dict(results_regression, orient='index', columns=["RMSE"])
print(reg_results)

print("\n================= CLASSIFICATION RESULTS =================")
clf_results = pd.DataFrame.from_dict(
    results_classification,
    orient='index',
    columns=["Accuracy", "F1-score"]
)
print(clf_results)


# Best Models
best_reg = reg_results["RMSE"].idxmin()
best_classi = clf_results["F1-score"].idxmax()

print("\n================= BEST MODELS =================")
print("Best Regression Model:", best_reg)
print("Best classification Model:", best_classi)

--- Starting Regression Task Setup (Ames Housing) ---
--- Starting Classification Task Setup (Credit Card Default) ---
Linear Regression (Normal Equation): RMSE = 34060.02
Linear Regression (Gradient Descent): RMSE = 30480.75
Decision Tree Regressor: RMSE = 40473.06
Random Forest Regressor: RMSE = 26096.97
KNN Regressor: RMSE = 34774.75
Logistic Regression: Accuracy = 0.8109, F1 = 0.3524
Decision Tree Classifier: Accuracy = 0.7283, F1 = 0.4059
Random Forest Classifier: Accuracy = 0.8197, F1 = 0.4772
KNN Classifier: Accuracy = 0.8037, F1 = 0.4257
SVM Classifier: Accuracy = 0.8208, F1 = 0.4499

================= REGRESSION RESULTS (RMSE) =================
                                              RMSE
Linear Regression (Normal Equation)   34060.016663
Linear Regression (Gradient Descent)  30480.751478
Decision Tree Regressor               40473.064611
Random Forest Regressor               26096.967920
KNN Regressor                         34774.750414

================= CLASSIFICATIO

In [ ]:
Resutls:
--- Starting Regression Task Setup (Ames Housing) ---
--- Starting Classification Task Setup (Credit Card Default) ---
Linear Regression (Normal Equation): RMSE = 34060.02
Linear Regression (Gradient Descent): RMSE = 30480.75
Decision Tree Regressor: RMSE = 40473.06
Random Forest Regressor: RMSE = 26096.97
KNN Regressor: RMSE = 34774.75
Logistic Regression: Accuracy = 0.8109, F1 = 0.3524
Decision Tree Classifier: Accuracy = 0.7283, F1 = 0.4059
Random Forest Classifier: Accuracy = 0.8197, F1 = 0.4772
KNN Classifier: Accuracy = 0.8037, F1 = 0.4257
SVM Classifier: Accuracy = 0.8208, F1 = 0.4499

================= REGRESSION RESULTS (RMSE) =================
                                              RMSE
Linear Regression (Normal Equation)   34060.016663
Linear Regression (Gradient Descent)  30480.751478
Decision Tree Regressor               40473.064611
Random Forest Regressor               26096.967920
KNN Regressor                         34774.750414

================= CLASSIFICATION RESULTS =================
                          Accuracy  F1-score
Logistic Regression       0.810867  0.352431
Decision Tree Classifier  0.728333  0.405890
Random Forest Classifier  0.819733  0.477185
KNN Classifier            0.803733  0.425673
SVM Classifier            0.820800  0.449857

================= BEST MODELS =================
Best Regression Model: Random Forest Regressor
Best Classification Model: Random Forest Classifier